# Object-Oriented Programming
**Topic:** Python Fundamentals

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


---
## What you'll explore

By the end of this demo you will be able to:

- **Define** a Python class with an `__init__` method, instance attributes, and instance methods
- **Explain** why classes bundle data and behavior together, and what problem that solves
- **Interpret** how scikit-learn's `.fit()`, `.predict()`, and `.transform()` pattern is a direct application of the class design you will learn here

---
## How we got here

In *03: Functions* we packaged logic into reusable units. A function, though, has no memory: every call starts fresh. A class adds memory to a function: it is a blueprint that creates objects, and those objects carry state between method calls. This is exactly the design that makes `model.fit(X_train)` followed later by `model.predict(X_test)` possible.

---
## Why this matters for data science

Every scikit-learn estimator is a class. `LinearRegression()` creates an instance; `.fit(X, y)` stores the learned weights in that instance; `.predict(X_new)` uses those stored weights to return predictions. Transformers like `StandardScaler` follow the same pattern: `fit()` stores the training mean and standard deviation, and `transform()` applies them. Once you understand this structure, the entire scikit-learn API becomes predictable and intuitive.

---
## The Four Principles of OOP

Object-Oriented Programming is built on four core principles. You will see all four appear naturally as we build through this notebook.

---

### 1. Encapsulation
**What it means:** Bundling data (attributes) and the functions that operate on that data (methods) together inside a single object. The object controls access to its own data.

**Plain English:** A `BankAccount` object knows its own balance and controls how that balance changes. Outside code cannot just set `account.balance = 1000000` directly — it must go through `deposit()` or `withdraw()`.

**Where you'll see it in this notebook:**
The `DataColumn` class in "Try it yourself" bundles `name` and `values` together and controls how they're accessed through methods like `mean()`.

**Where you'll see it in data science:**
`StandardScaler` encapsulates the training mean and standard deviation inside the object. You access them via `scaler.mean_` and `scaler.scale_` — you don't compute them yourself.

```python
# Encapsulation example
scaler = StandardScaler()
scaler.fit(X_train)
# mean_ and scale_ are now stored inside 
# the object — encapsulated
print(scaler.mean_)
print(scaler.scale_)
```

---

### 2. Abstraction
**What it means:** Hiding complex implementation details behind a simple interface. Users interact with the object through clean, simple method calls without needing to know how those methods work internally.

**Plain English:** You don't need to know the mathematics of ordinary least squares to call `model.fit(X, y)`. The complexity is hidden behind a simple method name.

**Where you'll see it in this notebook:**
`SimpleLinearRegressor.fit()` in the real-world example hides the OLS calculation behind a single method call. The user just calls `.fit()` — they don't see the covariance matrix calculation.

**Where you'll see it in data science:**
Every sklearn estimator abstracts away its algorithm. `RandomForestClassifier.fit()` hides hundreds of lines of tree-building code behind one method call.

```python
# Abstraction example
# You don't need to know how Random Forest 
# works internally to use it
model = RandomForestClassifier(n_estimators=100)
model.fit(X_train, y_train)  # complexity hidden
predictions = model.predict(X_test)
```

---

### 3. Inheritance
**What it means:** A new class can inherit attributes and methods from an existing class, extending or modifying its behavior without rewriting everything from scratch.

**Plain English:** A `SavingsAccount` class can inherit everything from `BankAccount` and just add an `apply_interest()` method. It doesn't need to rewrite `deposit()` or `withdraw()`.

**Where you'll see it in this notebook:**
The challenge section asks you to build a `BankAccount` class. You could extend it to a `SavingsAccount` using inheritance.

**Where you'll see it in data science:**
All sklearn estimators inherit from `BaseEstimator`. This gives every model `get_params()` and `set_params()` for free. Classifiers additionally inherit from `ClassifierMixin` which provides `.score()`.

```python
# Inheritance example
from sklearn.base import BaseEstimator

class MyCustomModel(BaseEstimator):
    # Inherits get_params(), set_params()
    # from BaseEstimator for free
    def __init__(self, alpha=1.0):
        self.alpha = alpha
    
    def fit(self, X, y):
        # your implementation
        return self
```

---

### 4. Polymorphism
**What it means:** Different classes can share the same method names but implement them differently. Code that calls `.fit()` works the same way regardless of which model is being used.

**Plain English:** `LinearRegression`, `RandomForest`, and `XGBoost` all have `.fit()` and `.predict()`. You can swap one for another without changing any other code.

**Where you'll see it in this notebook:**
The `SimpleLinearRegressor` has `.fit()` and `.predict()` — the same names as every sklearn model. This is polymorphism in action.

**Where you'll see it in data science:**
This is the most powerful principle for data science workflows. Because every model shares the same interface, you can write a single evaluation loop that works for any model:

```python
# Polymorphism example — same code, 
# any model
models = [
    LinearRegression(),
    RandomForestRegressor(),
    XGBRegressor()
]

for model in models:
    model.fit(X_train, y_train)  # same call
    score = model.score(X_test, y_test)  # same call
    print(f"{model.__class__.__name__}: {score:.3f}")
```

---

> **Key insight before you continue:**
> You have already been using all four principles every time you use sklearn — you just didn't have names for them yet. The rest of this notebook shows you how to build classes that use these same principles yourself.

---
## Try it yourself

In [ ]:
# ▶ Run this cell and observe the output.
# Then try changing the values and running again.

class DataColumn:
    def __init__(self, name, values):
        self.name   = name
        self.values = values

    def mean(self):
        return sum(self.values) / len(self.values)

    def __repr__(self):
        return f'DataColumn({self.name!r}, n={len(self.values)})'

col = DataColumn('salary', [52000, 75000, 88000, 61000, 94000])
print(col)
print(f'Mean: {col.mean():,.0f}')

In [ ]:
# ✏️ Your turn — modify this code:
# 1. Add a median() method to DataColumn
# 2. Add a missing_count() method that counts how many values are None
# 3. What happens when you print col before calling __repr__? Try removing it.

class DataColumn:
    def __init__(self, name, values):
        self.name   = name
        self.values = values

    def mean(self):
        return sum(self.values) / len(self.values)

    def __repr__(self):
        return f'DataColumn({self.name!r}, n={len(self.values)})'

col = DataColumn('age', [25, 30, 22, 45, 28])
print(col)
print(col.mean())

In [ ]:
# 🎯 Challenge:
# Create a BankAccount class with:
#   - balance attribute (set on __init__)
#   - deposit(amount) method that adds to balance
#   - withdraw(amount) method that subtracts — but prevents overdraft (balance below 0)
#   - __str__() method that returns 'Balance: $1,234.56'
# Hint: use an if statement in withdraw() to check whether the withdrawal is valid

# Your code here:

## The four principles in the examples above

| Principle | Where it appears in this notebook |
|-----------|----------------------------------|
| Encapsulation | `DataColumn` bundles `name` and `values` together — outside code accesses data through methods, not directly |
| Abstraction | `SimpleLinearRegressor.fit()` hides the OLS math — callers just call `.fit()` |
| Inheritance | `BankAccount` challenge can be extended to `SavingsAccount` without rewriting deposit/withdraw |
| Polymorphism | `SimpleLinearRegressor` uses `.fit()` and `.predict()` — same names as every sklearn model, swappable in any pipeline |

---
## What's happening?

A class is a blueprint for creating objects. Each object (instance) gets its own copy of the class's attributes and shares the class's methods.

| Concept | Syntax | What it means |
|---------|--------|--------------|
| Class definition | `class MyModel:` | Defines the blueprint |
| Constructor | `def __init__(self, param):` | Runs when an instance is created |
| Instance attribute | `self.coef_ = None` | Data stored on the specific object |
| Instance method | `def fit(self, X, y):` | Function that operates on `self` |
| Creating an instance | `model = MyModel()` | Builds one object from the blueprint |

```python
class SimpleLinearRegressor:
    """Mirrors the scikit-learn estimator API."""

    def __init__(self):
        self.coef_      = None   # set by fit()
        self.intercept_ = None   # set by fit()

    def fit(self, X, y):
        """Learn slope and intercept from training data."""
        self.coef_      = np.cov(X, y)[0, 1] / np.var(X)
        self.intercept_ = np.mean(y) - self.coef_ * np.mean(X)
        return self   # enables method chaining: model.fit(X, y).predict(X_test)

    def predict(self, X):
        """Apply the learned line to new data."""
        if self.coef_ is None:
            raise RuntimeError("Call fit() before predict().")
        return self.coef_ * X + self.intercept_
```

### The trailing underscore convention

Scikit-learn marks attributes learned during `fit()` with a trailing underscore: `coef_`, `mean_`, `n_features_in_`. Attributes set in `__init__` (hyperparameters) have no trailing underscore: `alpha`, `n_estimators`. This convention makes it immediately obvious whether an attribute was set by the user or learned from data.

Return to the widget and try calling predict before fit to see the error that enforces this workflow.

---
## A direct example: a class that remembers its data

A `ScoreStats` object stores a list of quiz scores on creation. Its methods compute from that stored list without needing the data passed in again.

- **Notice:** `self.scores` is set once in `__init__` and available in every method — this is why `quiz.mean()` works without any arguments
- **Notice:** Calling `quiz.above(60)` uses the same stored scores; the object remembers them between calls
- **Notice:** This is the same pattern as `StandardScaler`: `fit()` stores the training mean, and `transform()` uses it later without seeing the training data again

In [ ]:
class ScoreStats:
    def __init__(self, scores):
        self.scores = scores

    def mean(self):
        return sum(self.scores) / len(self.scores)

    def above(self, threshold):
        return [s for s in self.scores if s >= threshold]

quiz = ScoreStats([88, 52, 76, 91, 43, 67, 85, 59])
print(f"Mean:           {quiz.mean():.1f}")
print(f"Passing scores: {quiz.above(60)}")

colors = ["#4C72B0" if s >= 60 else "#E45756" for s in quiz.scores]
labels = [f"S{i+1}" for i in range(len(quiz.scores))]

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(labels, quiz.scores, color=colors)
ax.axhline(quiz.mean(), linestyle="--", color="black", linewidth=1,
           label=f"quiz.mean() = {quiz.mean():.1f}")
ax.set_title("ScoreStats: data stored in __init__, methods compute from self.scores", fontsize=12)
ax.set_xlabel("Student")
ax.set_ylabel("Score")
ax.legend()
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

---
## Real-world example: A custom LinearRegression class that mirrors scikit-learn

The chart below uses `SimpleLinearRegressor` to fit a line to advertising spend versus sales revenue for 80 simulated companies, then plot predictions. This is exactly what `sklearn.linear_model.LinearRegression` does internally.

Notice:

- **Notice:** The class stores `coef_` and `intercept_` on the instance after `fit()`, so `model.coef_` is readable directly, just like in scikit-learn
- **Notice:** Calling `predict()` on unseen X values produces the extrapolated line without the class needing to see the training data again, because the weights were stored in `self`
- **Notice:** The residuals (distances from points to the line) are the quantities that the fitting step minimized; the OLS estimator we implemented in two lines is mathematically equivalent to scikit-learn's implementation

> **Discussion question:** Scikit-learn's `LinearRegression` has a `fit_intercept=True` parameter in `__init__`. Where would you add that parameter to `SimpleLinearRegressor`, and how would you use it inside `fit()`?

In [ ]:
np.random.seed(42)

# ── SimpleLinearRegressor: custom class mirroring the sklearn API ──────────────
class SimpleLinearRegressor:
    """Minimal OLS linear regressor — same API as sklearn.linear_model.LinearRegression."""

    def __init__(self):
        self.coef_      = None
        self.intercept_ = None

    def fit(self, X, y):
        X_arr, y_arr    = np.asarray(X), np.asarray(y)
        self.coef_      = np.cov(X_arr, y_arr)[0, 1] / np.var(X_arr)
        self.intercept_ = y_arr.mean() - self.coef_ * X_arr.mean()
        return self

    def predict(self, X):
        if self.coef_ is None:
            raise RuntimeError("Call fit() before predict().")
        return self.coef_ * np.asarray(X) + self.intercept_

ad_spend = np.random.uniform(10, 200, 80)
revenue  = 3.5 * ad_spend + np.random.normal(0, 60, 80) + 50

model = SimpleLinearRegressor()
model.fit(ad_spend, revenue)

x_line = np.linspace(5, 210, 200)
y_hat  = model.predict(x_line)

fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(ad_spend, revenue, color="#4C72B0", alpha=0.65, s=50, label="Training data")
ax.plot(x_line, y_hat, color="#E45756", linewidth=2.5,
        label=f"model.predict()  coef_={model.coef_:.2f}  intercept_={model.intercept_:.1f}")
ax.set_title("SimpleLinearRegressor.fit() then .predict() — mirrors sklearn API", fontsize=13)
ax.set_xlabel("Ad Spend ($000s)")
ax.set_ylabel("Quarterly Revenue ($000s)")
ax.legend()
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()


### The scikit-learn estimator API in three method calls

| Method | What it does | Where data goes |
|--------|-------------|----------------|
| `__init__(hyperparams)` | Creates the instance, stores configuration | Stored as plain attributes (e.g., `self.alpha`) |
| `.fit(X, y)` | Learns from training data, stores learned parameters | Stored as trailing-underscore attributes (e.g., `self.coef_`) |
| `.predict(X)` / `.transform(X)` | Applies stored parameters to new data | Returns an array; does not modify `self` |
| `.fit_transform(X)` | Calls `fit()` then `transform()` in one step | Convenience for transformers only |
| `.score(X, y)` | Evaluates the fitted model on labeled data | Returns a scalar (R² for regressors, accuracy for classifiers) |

---
## Key takeaway

> **A class bundles data and behavior into an object that remembers state between method calls; this is exactly why scikit-learn can separate .fit() from .predict() and why learning the API of one estimator teaches you all of them.**

---
*Next up: Error Handling — what happens when your pipeline encounters bad data, and how to write code that fails gracefully instead of crashing silently*